In [ ]:
%pip install python-dotenv langchain-upstage


In [ ]:
%pip install langchain langchain-community langchain-text-splitters langchain-pinecone docx2txt

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

ai_message = llm.invoke("한국에 대해 3줄로 설명해줘")

ai_message.content

In [ ]:
# docx파일 load, split and saving in the vector storage

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./접구선통기술기준_20251128.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [7]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name_gl = 'groundandlinelaw-index'
database1 = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name_gl)

In [8]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./방송통신설비기술기준_20240719.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name_bc = 'broadcastandcomlaw-index'
database2 = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name_bc)

In [ ]:
%pip install langchain-pinecone

In [8]:
# from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name_gl = 'groundandlinelaw-index'
index_name_bc = 'broadcastandcomlaw-index'

groundline_db = PineconeVectorStore(index_name=index_name_gl, embedding=embedding)
broadcom_db = PineconeVectorStore(index_name=index_name_bc, embedding=embedding)

In [21]:
query = '통신설비의 접지저항은 얼마입니까?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
docs1 = groundline_db.similarity_search_with_score(query, k=5)
docs2 = broadcom_db.similarity_search_with_score(query, k=5)

all_docs = docs1 + docs2

all_docs = sorted(all_docs, key=lambda x:x[1],reverse=True)

In [ ]:
all_docs